# CDC PLACES National Census-Tract Cleaner

This Colab notebook converts the raw **CDC PLACES: Census Tract Data (GIS Friendly Format), 2025 release** CSV into:

`national_places_lace_clean.csv`

That cleaned file can then be uploaded into the separate national diabetes, heat, and air-conditioning analysis notebook.

## What this cleaner does

- Uploads the raw CDC PLACES CSV
- Detects the relevant columns automatically
- Keeps one row per U.S. census tract
- Extracts diabetes prevalence and useful geographic fields
- Standardizes tract GEOIDs to 11 digits
- Removes duplicate and invalid tract records
- Exports and downloads `national_places_lace_clean.csv`


In [ ]:
# Import packages
import io
import re
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Upload the raw CDC PLACES CSV

Upload the file whose name looks similar to:

`PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2025_release.csv`


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise ValueError("Please upload a CSV file.")

raw_filename = csv_names[0]
print("Using:", raw_filename)

raw = pd.read_csv(
    io.BytesIO(uploaded[raw_filename]),
    dtype=str,
    low_memory=False
)

print("Raw shape:", raw.shape)
display(raw.head())


## 2. Inspect and normalize column names


In [ ]:
def normalize_name(name):
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

raw.columns = [normalize_name(c) for c in raw.columns]

print("Normalized columns:")
for c in raw.columns:
    print(c)


## 3. Automatically identify the required fields

The PLACES GIS-friendly release normally includes fields such as `GEOID`, state, county, tract, population, and measure-specific prevalence columns. This cell searches for them even if the precise capitalization differs.


In [ ]:
def first_existing(candidates, required=False):
    lower_map = {c.lower(): c for c in raw.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise KeyError(
            "Could not find any of these required columns: "
            + ", ".join(candidates)
        )
    return None

geoid_col = first_existing([
    "GEOID", "LocationID", "Location_ID", "CensusTract", "Census_Tract"
], required=True)

state_abbr_col = first_existing([
    "StateAbbr", "State_Abbr", "StateAbbreviation", "State_Abbreviation"
])

state_name_col = first_existing([
    "StateDesc", "State_Desc", "StateName", "State_Name"
])

county_name_col = first_existing([
    "CountyName", "County_Name", "County"
])

county_fips_col = first_existing([
    "CountyFIPS", "County_FIPS", "CountyFips"
])

tract_name_col = first_existing([
    "LocationName", "Location_Name", "TractName", "Tract_Name"
])

population_col = first_existing([
    "TotalPopulation", "Total_Population", "PopulationCount",
    "Population_Count", "Population2010", "Population_2010"
])

# Search broadly for the diabetes crude-prevalence field.
diabetes_candidates = [
    c for c in raw.columns
    if "diabetes" in c.lower()
    and any(token in c.lower() for token in ["crude", "prev", "prevalence"])
]

# Prefer a field containing both crude and prevalence.
preferred = [
    c for c in diabetes_candidates
    if "crude" in c.lower()
    and any(token in c.lower() for token in ["prev", "prevalence"])
]

if preferred:
    diabetes_col = preferred[0]
elif diabetes_candidates:
    diabetes_col = diabetes_candidates[0]
else:
    raise KeyError(
        "No diabetes prevalence column was detected. "
        "Review the printed columns and confirm this is the GIS-friendly PLACES tract file."
    )

detected = {
    "GEOID": geoid_col,
    "StateAbbr": state_abbr_col,
    "StateName": state_name_col,
    "CountyName": county_name_col,
    "CountyFIPS": county_fips_col,
    "TractName": tract_name_col,
    "TotalPopulation": population_col,
    "DIABETES_CrudePrev": diabetes_col
}

print("Detected fields:")
for clean_name, source_name in detected.items():
    print(f"{clean_name:22s} <- {source_name}")


## 4. Build the cleaned national tract file


In [ ]:
clean = pd.DataFrame()

# Standardize the 11-digit census-tract GEOID.
clean["GEOID"] = (
    raw[geoid_col]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(11)
)

if state_abbr_col:
    clean["StateAbbr"] = raw[state_abbr_col].astype(str).str.strip().str.upper()
else:
    clean["StateAbbr"] = np.nan

if state_name_col:
    clean["StateName"] = raw[state_name_col].astype(str).str.strip()
else:
    clean["StateName"] = np.nan

if county_name_col:
    clean["CountyName"] = raw[county_name_col].astype(str).str.strip()
else:
    clean["CountyName"] = np.nan

if county_fips_col:
    clean["CountyFIPS"] = (
        raw[county_fips_col]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(5)
    )
else:
    clean["CountyFIPS"] = clean["GEOID"].str[:5]

if tract_name_col:
    clean["TractName"] = raw[tract_name_col].astype(str).str.strip()
else:
    clean["TractName"] = np.nan

if population_col:
    clean["TotalPopulation"] = pd.to_numeric(
        raw[population_col].str.replace(",", "", regex=False),
        errors="coerce"
    )
else:
    clean["TotalPopulation"] = np.nan

clean["DIABETES_CrudePrev"] = pd.to_numeric(
    raw[diabetes_col]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

# Derive state and tract FIPS components.
clean["StateFIPS"] = clean["GEOID"].str[:2]
clean["TractFIPS"] = clean["GEOID"].str[5:11]

# Retain valid census tract identifiers only.
clean = clean[
    clean["GEOID"].str.fullmatch(r"\d{11}", na=False)
].copy()

# Remove records without a usable diabetes estimate.
clean = clean[
    clean["DIABETES_CrudePrev"].between(0, 100, inclusive="both")
].copy()

# Remove exact duplicate GEOIDs.
before_dupes = len(clean)
clean = clean.drop_duplicates(subset="GEOID", keep="first").copy()
removed_dupes = before_dupes - len(clean)

# Reorder columns.
column_order = [
    "GEOID", "StateFIPS", "StateAbbr", "StateName",
    "CountyFIPS", "CountyName", "TractFIPS", "TractName",
    "TotalPopulation", "DIABETES_CrudePrev"
]
clean = clean[column_order].sort_values("GEOID").reset_index(drop=True)

print("Clean shape:", clean.shape)
print("Duplicate GEOIDs removed:", removed_dupes)
display(clean.head())


## 5. Run quality checks


In [ ]:
quality = pd.DataFrame({
    "check": [
        "Number of tract rows",
        "Unique GEOIDs",
        "Missing diabetes estimates",
        "Minimum diabetes prevalence",
        "Median diabetes prevalence",
        "Maximum diabetes prevalence",
        "States/territories represented"
    ],
    "value": [
        len(clean),
        clean["GEOID"].nunique(),
        clean["DIABETES_CrudePrev"].isna().sum(),
        clean["DIABETES_CrudePrev"].min(),
        clean["DIABETES_CrudePrev"].median(),
        clean["DIABETES_CrudePrev"].max(),
        clean["StateFIPS"].nunique()
    ]
})

display(quality)

assert clean["GEOID"].str.len().eq(11).all(), "Some GEOIDs are not 11 digits."
assert clean["GEOID"].is_unique, "Duplicate tract GEOIDs remain."
assert clean["DIABETES_CrudePrev"].between(0, 100).all(), "Invalid diabetes prevalence values remain."

print("All required validation checks passed.")


## 6. Export and download the cleaned file


In [ ]:
output_filename = "national_places_lace_clean.csv"
clean.to_csv(output_filename, index=False)

print(f"Created {output_filename}")
print(f"Rows: {len(clean):,}")
print(f"Columns: {len(clean.columns)}")

files.download(output_filename)


## Next step

Open `Diabetes_Heat_AC_National_Study_Colab.ipynb`, run it in Google Colab, and upload the newly downloaded:

`national_places_lace_clean.csv`
